# Lab 2 — CIFAR-10 transfer learning

Chạy trên **VS Code + extension Google Colab**: chọn kernel Colab có **GPU T4**, rồi chạy lần lượt từ trên xuống. Máy Colab tự lấy code từ GitHub, không cần upload Drive.

- Giữ VS Code mở đến khi chạy xong; mỗi cell lỗi sẽ dừng, không chạy tiếp với dữ liệu thiếu.
- Không chạy lại cell train hoặc evaluation đã xong.
- Chạy xong: cell cuối chép `checkpoints/` và `reports/` sang Google Drive. Tải thư mục đó về, chép vào `Lab 2/`, rồi **Ctrl+S** để lưu notebook có output.

Thứ tự bước và danh sách kết quả cần có: `docs/training.md` mục 9.

## 1. Lấy code từ GitHub

In [ ]:
import os, sys, subprocess
from pathlib import Path

BRANCH = 'QuocAn'                 # nhánh chứa code Lab 2 cần chạy
OPERATOR = 'Nguyễn Quốc An'       # người chạy, ghi vào metrics.json
REPO = 'https://github.com/QuocAn118/Lab.git'
ROOT = Path('/content/Lab')

if not ROOT.exists():
    # Bỏ tải file LFS (checkpoint cũ của các lab) vì sẽ train mới.
    subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, REPO, str(ROOT)], check=True,
                   env={**os.environ, 'GIT_LFS_SKIP_SMUDGE': '1'})
PROJECT_DIR = ROOT / 'Lab 2'
os.chdir(PROJECT_DIR)
os.environ.update(PYTHONPATH=str(PROJECT_DIR / 'src'), LAB2_OPERATOR=OPERATOR, PYTHONUTF8='1')
REPORTS = PROJECT_DIR / 'reports'
print(subprocess.run(['git', 'log', '-1', '--format=Commit %h — %s'], capture_output=True, text=True).stdout)


def run(*cmd):
    # In log từng dòng ngay khi lệnh chạy; lệnh lỗi thì dừng notebook.
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='')
    if proc.wait():
        raise RuntimeError(f'Lệnh lỗi (exit {proc.returncode}): {" ".join(map(str, cmd))}')


def lab(module, *args):
    run(sys.executable, '-m', module, *args)

## 2. Cài thư viện và kiểm tra GPU

In [ ]:
run(sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt')
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU — chậm; KPI GPU không đo được')

## 3. Kiểm tra offline — dữ liệu giả, không phải số liệu thực nghiệm

In [ ]:
lab('pytest', 'tests', '--ignore=tests/test_smoke.py', '-q')

## 4. Bước 5 — tải và chia CIFAR-10; torchvision kiểm tra MD5

In [ ]:
import urllib.request
archive = Path('data/cifar-10-python.tar.gz')
archive.parent.mkdir(parents=True, exist_ok=True)
if not archive.exists():
    urllib.request.urlretrieve('https://huggingface.co/datasets/MIT-OL-AI-D/cifar-10-python/resolve/main/cifar-10-python.tar.gz', archive)
lab('lab_2.data')

## 5. Bước 2–4 — model pre-trained, kiến trúc và tham số

In [ ]:
lab('lab_2.model')

## 6. Smoke thật — 2 batch, 3 cấu hình; chỉ tiếp tục khi đều pass

In [ ]:
lab('pytest', 'tests/test_smoke.py', '-q', '-s')

## Baseline — resnet18_baseline

In [ ]:
lab('lab_2.training', '--run', 'resnet18_baseline')

## Exercise 1 — đổi model — densenet121

In [ ]:
lab('lab_2.training', '--run', 'densenet121')

## Exercise 2 — đổi LR — resnet18_lr1e-2

In [ ]:
lab('lab_2.training', '--run', 'resnet18_lr1e-2')

## Exercise 2 — đổi batch — resnet18_bs128

In [ ]:
lab('lab_2.training', '--run', 'resnet18_bs128')

## Đánh giá test một lần — model được chọn bằng validation

In [ ]:
lab('lab_2.evaluation')

## Biểu đồ đánh giá

In [ ]:
from IPython.display import display, Image
for name in ['val_accuracy_by_run.png', 'loss_curve_best.png', 'confusion_matrix.png']:
    display(Image(filename=str(REPORTS / 'figures' / name)))

## Demo — ba ảnh đầu của tập test; có thể dùng --image với ảnh của bạn

In [ ]:
lab('lab_2.demo')
display(Image(filename=str(REPORTS / 'figures/demo_probabilities.png')))

## Bảng kết quả và môi trường — từ reports/metrics.json

In [ ]:
import json
report = json.loads((REPORTS / 'metrics.json').read_text(encoding='utf-8'))
for section in ['comparison', 'kpi', 'test_metrics', 'per_class_accuracy']:
    print(section)
    display(report['evaluation'][section])
print('Best run:', report['evaluation']['best_run'])
for name, summary in report['training']['runs'].items():
    print(name, summary['operator'], summary['started_at'], summary['finished_at'], summary['device'], summary['library_versions'], summary['code_commit'])

## Exercise 3 — TensorBoard

Chụp biểu đồ cả 4 run và lưu thành `reports/figures/tensorboard.png`. Nếu khung TensorBoard không hiện trong VS Code: sau khi tải kết quả về, chạy `tensorboard --logdir reports/runs` trên máy rồi chụp.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir reports/runs

## Kiểm tra đủ kết quả, rồi chép sang Google Drive

In [ ]:
runs = ['resnet18_baseline', 'densenet121', 'resnet18_lr1e-2', 'resnet18_bs128']
expected = [f'checkpoints/{r}/{f}.pth' for r in runs for f in ('final_epoch', 'best_val_loss', 'best_val_acc')]
expected += ['checkpoints/best.pth', 'reports/metrics.json']
expected += [f'reports/figures/{f}.png' for f in ('val_accuracy_by_run', 'loss_curve_best', 'confusion_matrix', 'demo_probabilities')]
missing = [p for p in expected if not (PROJECT_DIR / p).is_file()]
missing += [f'reports/runs/{r}/' for r in runs if not list((REPORTS / 'runs' / r).glob('events.out.tfevents*'))]
print('Thiếu:', missing) if missing else print('Đủ 13 checkpoint, metrics.json, 4 biểu đồ, log TensorBoard 4 run')

import shutil, datetime
from google.colab import drive
drive.mount('/content/drive')
dest = Path('/content/drive/MyDrive/Lab2_ket_qua') / datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
for name in ['checkpoints', 'reports']:
    shutil.copytree(PROJECT_DIR / name, dest / name)
print('Đã chép sang', dest)